# Exercice 07 - Ingestion PostgreSQL vers Bronze

## Objectifs
- Extraire des donnees de PostgreSQL
- Sauvegarder dans la couche Bronze de MinIO
- Organiser les donnees par date d'ingestion
- Creer un pipeline d'ingestion reutilisable

---

## 1. Architecture d'ingestion

```
+----------------+                    +------------------------+
|                |                    |        MinIO           |
|   PostgreSQL   |     SPARK          |                        |
|                |  =============>    |  +------------------+  |
|  +----------+  |                    |  |     BRONZE       |  |
|  | customers|  |                    |  +------------------+  |
|  | products |  |                    |  | /customers/      |  |
|  | orders   |  |                    |  |   /2024-01-15/   |  |
|  | ...      |  |                    |  | /products/       |  |
|  +----------+  |                    |  |   /2024-01-15/   |  |
|                |                    |  | /orders/         |  |
+----------------+                    |  |   /2024-01-15/   |  |
                                      |  +------------------+  |
                                      +------------------------+

Format Bronze : donnees brutes en Parquet
Organisation  : /table/date_ingestion/
```

## 2. Configuration Spark pour PostgreSQL et MinIO

In [1]:
from pyspark.sql import SparkSession
from datetime import datetime

# Creer la SparkSession avec les configurations
spark = SparkSession.builder \
    .appName("Ingestion PostgreSQL vers Bronze") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0,org.apache.hadoop:hadoop-aws:3.4.1,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("Spark configure pour PostgreSQL et MinIO")

Spark configure pour PostgreSQL et MinIO


In [ ]:
# Configuration PostgreSQL
jdbc_url = "jdbc:postgresql://postgres:5432/app"
jdbc_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

# Date d'ingestion pour organiser les fichiers
date_ingestion = datetime.now().strftime("%Y-%m-%d")
print(f"Date d'ingestion : {date_ingestion}")

## 3. Ingerer une table simple

In [ ]:
# Lire la table customers depuis PostgreSQL
df_customers = spark.read.jdbc(
    url=jdbc_url,
    table="customers",
    properties=jdbc_properties
)

print(f"Customers : {df_customers.count()} lignes")
df_customers.show(5)

In [ ]:
# Sauvegarder dans Bronze
chemin_bronze = f"s3a://bronze/customers/{date_ingestion}"

df_customers.write \
    .mode("overwrite") \
    .parquet(chemin_bronze)

print(f"Sauvegarde reussie : {chemin_bronze}")

In [ ]:
# Verifier la sauvegarde
df_check = spark.read.parquet(chemin_bronze)
print(f"Verification : {df_check.count()} lignes lues depuis Bronze")

## 4. Fonction d'ingestion reutilisable

In [ ]:
def ingerer_table(nom_table, date=None):
    """
    Ingere une table PostgreSQL vers le bucket Bronze.
    
    Args:
        nom_table: Nom de la table a ingerer
        date: Date d'ingestion (defaut: aujourd'hui)
    
    Returns:
        Nombre de lignes ingerees
    """
    if date is None:
        date = datetime.now().strftime("%Y-%m-%d")
    
    # Lire depuis PostgreSQL
    df = spark.read.jdbc(
        url="jdbc:postgresql://postgres:5432/app",
        table=nom_table,
        properties={
            "user": "postgres",
            "password": "postgres",
            "driver": "org.postgresql.Driver"
        }
    )
    
    # Sauvegarder dans Bronze
    chemin = f"s3a://bronze/{nom_table}/{date}"
    df.write.mode("overwrite").parquet(chemin)
    
    nb_lignes = df.count()
    print(f"[OK] {nom_table} : {nb_lignes} lignes -> {chemin}")
    
    return nb_lignes

In [ ]:
# Tester la fonction
ingerer_table("products")

## 5. Ingerer toutes les tables Northwind

In [ ]:
# Liste des tables a ingerer
tables_northwind = [
    "categories",
    "customers",
    "employees",
    "orders",
    "order_details",
    "products",
    "shippers",
    "suppliers",
    "region",
    "territories"
]

print(f"Tables a ingerer : {len(tables_northwind)}")

In [ ]:
# Ingerer toutes les tables
resultats = {}

print("=" * 50)
print("INGESTION NORTHWIND VERS BRONZE")
print("=" * 50)

for table in tables_northwind:
    try:
        nb_lignes = ingerer_table(table, date_ingestion)
        resultats[table] = {"status": "OK", "lignes": nb_lignes}
    except Exception as e:
        print(f"[ERREUR] {table} : {e}")
        resultats[table] = {"status": "ERREUR", "erreur": str(e)}

print("=" * 50)
print("INGESTION TERMINEE")
print("=" * 50)

In [ ]:
# Resume de l'ingestion
print("\nResume :")
total_lignes = 0
for table, info in resultats.items():
    if info["status"] == "OK":
        print(f"  {table}: {info['lignes']} lignes")
        total_lignes += info["lignes"]
    else:
        print(f"  {table}: ERREUR")

print(f"\nTotal : {total_lignes} lignes ingerees")

## 6. Ajouter des metadonnees d'ingestion

In [ ]:
from pyspark.sql import functions as F

def ingerer_table_avec_metadata(nom_table, date=None):
    """
    Ingere une table avec des colonnes de metadata.
    """
    if date is None:
        date = datetime.now().strftime("%Y-%m-%d")
    
    timestamp_ingestion = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Lire depuis PostgreSQL
    df = spark.read.jdbc(
        url="jdbc:postgresql://postgres:5432/app",
        table=nom_table,
        properties={
            "user": "postgres",
            "password": "postgres",
            "driver": "org.postgresql.Driver"
        }
    )
    
    # Ajouter les metadonnees
    df = df.withColumn("_source", F.lit("postgresql")) \
           .withColumn("_table", F.lit(nom_table)) \
           .withColumn("_ingestion_date", F.lit(date)) \
           .withColumn("_ingestion_timestamp", F.lit(timestamp_ingestion))
    
    # Sauvegarder dans Bronze
    chemin = f"s3a://bronze/{nom_table}/{date}"
    df.write.mode("overwrite").parquet(chemin)
    
    nb_lignes = df.count()
    print(f"[OK] {nom_table} : {nb_lignes} lignes (avec metadata) -> {chemin}")
    
    return nb_lignes

In [ ]:
# Tester avec metadata
ingerer_table_avec_metadata("categories")

# Verifier les metadonnees
df_test = spark.read.parquet(f"s3a://bronze/categories/{date_ingestion}")
df_test.select("category_id", "category_name", "_source", "_table", "_ingestion_date").show()

## 7. Verifier le contenu de Bronze

In [ ]:
# Lister les fichiers dans Bronze
print("Contenu du bucket Bronze :")
print("=" * 50)

for table in tables_northwind:
    try:
        chemin = f"s3a://bronze/{table}/{date_ingestion}"
        df = spark.read.parquet(chemin)
        print(f"{table}: {df.count()} lignes")
    except:
        print(f"{table}: non trouve")

---

## Exercice

**Objectif** : Creer un script d'ingestion complet

**Consigne** :
1. Creez une fonction `ingestion_complete()` qui :
   - Ingere toutes les tables avec metadonnees
   - Affiche un rapport final
   - Retourne un dictionnaire avec les statistiques

2. Ajoutez une colonne `_nb_colonnes` aux metadonnees

A vous de jouer :

In [ ]:
def ingestion_complete():
    """
    Ingere toutes les tables Northwind vers Bronze.
    
    Returns:
        dict: Statistiques d'ingestion
    """
    # TODO: Implementer la fonction
    pass

---

## Resume

Dans ce notebook, vous avez appris :
- Comment **extraire des donnees** de PostgreSQL avec Spark
- Comment **sauvegarder** les donnees dans MinIO (Bronze)
- Comment **organiser** les donnees par date d'ingestion
- Comment **ajouter des metadonnees** pour la tracabilite
- Comment creer un **pipeline d'ingestion** reutilisable

### Prochaine etape
Dans le prochain notebook, nous apprendrons a ingerer des donnees depuis le Web (API REST).